<a href="https://colab.research.google.com/github/Deepakrewa/hello-world/blob/master/URLScreenshots09Oct2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install playwright pillow
!playwright install chromium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 21.2 MB/s eta 0:00:00
173.7 MiB [] 0% 0.0s173.7 MiB [] 0% 34.5s173.7 MiB [] 0% 15.7s173.7 MiB [] 0% 8.4s173.7 MiB [] 1% 4.8s173.7 MiB [] 2% 3.6s173.7 MiB [] 3% 2.7s173.7 MiB [] 4% 2.4s173.7 MiB [] 5% 2.5s173.7 MiB [] 6% 2.5s173.7 MiB [] 6% 2.4s173.7 MiB [] 7% 2.4s173.7 MiB [] 8% 2.2s173.7 MiB [] 9% 2.2s173.7 MiB [] 9% 2.3s173.7 MiB [] 11% 2.0s173.7 MiB [] 12% 2.0s173.7 MiB [] 13% 1.9s173.7 MiB [] 14% 1.9s173.7 MiB [] 16% 1.8s173.7 MiB [] 17% 1.6s173.7 MiB [] 19% 1.6s173.7 MiB [] 20% 1.5s173.7 MiB [] 21% 1.5s173.7 MiB [] 23% 1.4s173.7 MiB [] 24% 1.3s173.7 MiB [] 26% 1.3s173.7 MiB [] 27% 1.3s173.7 MiB [] 28% 1.3s173.7 MiB [] 30% 1.2s173.7 MiB [] 32% 1.1s173.7 MiB [] 33% 1.1s173.7 MiB [] 35% 1.1s173.7 MiB [] 36% 1.0s173.7 MiB [] 38% 1.0s173.7 MiB [] 39% 0.9s173.7 MiB [] 41% 0.9s173.7 MiB [] 43% 0.9s173.7 MiB [] 44% 0.8s173.7 MiB [] 46% 0.8s173.7 MiB [] 47% 0.8s173.7 MiB [] 50% 0.7s173.7 MiB [] 52% 0.7s173.7 MiB [] 53% 0.6s173.7 MiB 

In [ ]:
import asyncio
import csv
import hashlib
import os
import re
import time
import textwrap
from urllib.parse import urlparse
from datetime import datetime, timezone
from PIL import Image, ImageDraw, ImageFont
from playwright.async_api import async_playwright, TimeoutError as PWTimeout
import shutil
from google.colab import files
import zipfile

# Config
INPUT_FILE = "/content/print"
OUT_DIR = "screenshots"
SUCCESS_DIR = os.path.join(OUT_DIR, "success")
FAILURE_DIR = os.path.join(OUT_DIR, "failures")
REPORT_CSV = "report.csv"
ZIP_FILE = "screenshots_bundle.zip"

MAX_RETRIES = 3
NAV_TIMEOUT_MS = 20000
NETWORK_IDLE_MS = 5000
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
)

# Ensure folders exist
os.makedirs(SUCCESS_DIR, exist_ok=True)
os.makedirs(FAILURE_DIR, exist_ok=True)


def normalize_url(raw: str) -> str:
    """Normalize URL and default to HTTPS if no scheme provided."""
    raw = raw.strip()
    if not raw:
        return ""
    # If user already added http:// or https:// — keep it
    if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", raw):
        return raw
    # Prefer HTTPS by default
    return "https://" + raw


def slugify_filename(url: str, idx: int) -> str:
    p = urlparse(url)
    host = p.netloc.replace(":", "_")
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()[:10]
    return f"{idx:03d}_{host}_{h}.png"


def make_placeholder(path: str, message: str):
    """Create an error screenshot placeholder with message."""
    img = Image.new("RGB", (1200, 300), color=(255, 230, 230))
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()
    wrapped = textwrap.wrap(message, width=100)
    y = 50
    for line in wrapped:
        draw.text((10, y), line, fill=(180, 0, 0), font=font)
        y += 20
    img.save(path)


async def attempt_visit(ctx, url: str, idx: int, attempt: int):
    """Try visiting a URL once."""
    page = await ctx.new_page()
    t0 = time.perf_counter()
    record = {
        "index": idx,
        "input_url": url,
        "final_url": "",
        "status_code": "",
        "title": "",
        "screenshot_path": "",
        "timestamp": "",
        "load_time_s": "",
        "ok": "0",
        "error": "",
        "attempt": attempt,
    }
    try:
        resp = await page.goto(url, wait_until="domcontentloaded", timeout=NAV_TIMEOUT_MS)
        try:
            await page.wait_for_load_state("networkidle", timeout=NETWORK_IDLE_MS)
        except PWTimeout:
            pass

        record["final_url"] = page.url
        record["status_code"] = str(resp.status) if resp else ""
        record["title"] = await page.title()
        record["timestamp"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

        # success screenshot path
        shot_name = slugify_filename(record["final_url"] or url, idx)
        shot_path = os.path.join(SUCCESS_DIR, shot_name)

        # Take full page screenshot
        await page.screenshot(path=shot_path, full_page=True)

        # Add header info
        img = Image.open(shot_path)
        width, height = img.size
        bar_height = 65
        new_img = Image.new("RGB", (width, height + bar_height), color=(240, 240, 240))
        new_img.paste(img, (0, bar_height))

        draw = ImageDraw.Draw(new_img)
        font = ImageFont.load_default()
        header_text = (
            f"URL: {record['final_url']} | Status: {record['status_code']} "
            f"| Time: {record['timestamp']} | Attempt: {attempt}"
        )
        wrapped = textwrap.wrap(header_text, width=120)
        y = 10
        for line in wrapped:
            draw.text((10, y), line, fill=(0, 0, 0), font=font)
            y += 15

        new_img.save(shot_path)

        record["screenshot_path"] = shot_path
        record["ok"] = "1"
    except Exception as e:
        record["error"] = repr(e)
        record["timestamp"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    finally:
        record["load_time_s"] = f"{time.perf_counter() - t0:.2f}"
        await page.close()
    return record


async def visit_with_retries(ctx, url: str, idx: int):
    """Retry visiting a URL up to MAX_RETRIES times.
       Tries HTTPS first, then falls back to HTTP if HTTPS fails."""
    original_url = url
    for attempt in range(1, MAX_RETRIES + 1):
        rec = await attempt_visit(ctx, url, idx, attempt)
        if rec["ok"] == "1":
            return rec

        # If first HTTPS attempt failed, fallback to HTTP once
        if attempt == 1 and url.startswith("https://"):
            fallback = "http://" + url[len("https://"):]
            print(f"[{idx}] HTTPS failed, retrying with HTTP: {fallback}")
            url = fallback

    # if all failed → save placeholder screenshot
    shot_name = slugify_filename(original_url, idx)
    shot_path = os.path.join(FAILURE_DIR, shot_name)
    make_placeholder(shot_path, f"Failed after {MAX_RETRIES} attempts: {original_url}\nLast error: {rec['error']}")
    rec["screenshot_path"] = shot_path
    return rec


async def main():
    # Load and normalize URLs
    with open(INPUT_FILE, "r") as f:
        raw_urls = [line.strip() for line in f if line.strip()]
    urls = [normalize_url(u) for u in raw_urls if normalize_url(u)]
    results = []

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        ctx = await browser.new_context(ignore_https_errors=True, user_agent=USER_AGENT)

        for idx, url in enumerate(urls, start=1):
            print(f"[{idx}] Visiting: {url}")
            rec = await visit_with_retries(ctx, url, idx)
            results.append(rec)

        await ctx.close()
        await browser.close()

    # Save CSV report
    fieldnames = [
        "index", "input_url", "final_url", "status_code",
        "title", "timestamp", "screenshot_path",
        "load_time_s", "ok", "error", "attempt"
    ]
    with open(REPORT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    # Zip everything
    if os.path.exists(ZIP_FILE):
        os.remove(ZIP_FILE)
    with zipfile.ZipFile(ZIP_FILE, "w") as zf:
        zf.write(REPORT_CSV)
        for folder in [SUCCESS_DIR, FAILURE_DIR]:
            for root, _, files_list in os.walk(folder):
                for file in files_list:
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, OUT_DIR)
                    zf.write(full_path, arcname=os.path.join("screenshots", arcname))

    print(f"✅ Done! All saved into '{ZIP_FILE}'")
    files.download(ZIP_FILE)


# Run in Colab
await main()


[1] Visiting: https://modgovindia.space
[1] HTTPS failed, retrying with HTTP: http://modgovindia.space
[2] Visiting: https://modgovindia.com
[3] Visiting: https://mygov.cloud
[4] Visiting: https://dgqadefence.in
[4] HTTPS failed, retrying with HTTP: http://dgqadefence.in
[5] Visiting: https://email.gov.in.publications.it
[5] HTTPS failed, retrying with HTTP: http://email.gov.in.publications.it
[6] Visiting: https://govtdrdo.com
[7] Visiting: https://indiapost.gov.fcxajcp.top.in
[8] Visiting: https://mea.gov-in.com
[9] Visiting: https://mha-gov.ink
[9] HTTPS failed, retrying with HTTP: http://mha-gov.ink
✅ Done! All saved into 'screenshots_bundle.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# New Section